# Generalizable ECG Classification

In [ ]:
# Imports
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

In [ ]:
import matplotlib
matplotlib.use("inline")  
%matplotlib inline

In [ ]:
# Loading data directories
LTDB_DIR  = Path("../raw_data/mit-bih-long-term-ecg-database-1.0.0")
PTBXL_DIR = Path("../raw_data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")

## Data Preprocessing and EDA

In [ ]:
ltdb_records = [p.stem for p in LTDB_DIR.glob("*.hea") if not p.name.endswith(".hea-")]

In [ ]:
# Annotation symbols across all records

all_symbols = []
for name in ltdb_records:
    ann = wfdb.rdann(str(LTDB_DIR / name), "atr")
    all_symbols.extend(ann.symbol)

counts = Counter(all_symbols)
print("All annotation symbols and counts:")
for sym, cnt in counts.most_common():
    print(f"  {sym:4s}  {cnt:>8,}")

### Annotation Symbols and Meanings

- **N** → Normal beat  
- **V** → Ventricular ectopic beat (PVC)  
- **F** → Fusion beat (normal + ventricular)  
- **S** → Supraventricular ectopic beat  
- **~** → Noise / signal artifact  
- **J** → Junctional beat  
- **a** → Aberrated atrial beat

Association for the Advancement of Medical Instrumentation (AAMI) defines a standard grouping of ECG beat types to simplify classification

In [ ]:
# AAMI mapping
AAMI_MAP = {
    "N":"N","L":"N","R":"N","e":"N","j":"N",
    "A":"S","a":"S","J":"S","S":"S",
    "V":"V","E":"V",
    "F":"F",
    "/":"Q","f":"Q","Q":"Q"
}

aami_counts = Counter()
for sym, cnt in counts.items():
    if sym in AAMI_MAP:
        aami_counts[AAMI_MAP[sym]] += cnt

print("AAMI class distribution:")
for cls, cnt in aami_counts.most_common():
    print(f"  {cls}  {cnt:>8,}")

In [ ]:
# Plotting a single beat

rec = wfdb.rdrecord(str(LTDB_DIR / ltdb_records[0]))
ann = wfdb.rdann(str(LTDB_DIR / ltdb_records[0]), "atr")
sig = rec.p_signal

# find first N beat with enough room on both sides
r = None
for sample, sym in zip(ann.sample, ann.symbol):
    if sym == "N" and sample > 100 and sample + 150 < len(sig):
        r = sample
        break

print("R-peak used:", r)
window = sig[r-100 : r+150]

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
for i, name in enumerate(rec.sig_name):
    axes[i].plot(window[:, i], lw=0.9)
    axes[i].axvline(100, color="red", lw=0.7, ls="--", label="R-peak")
    axes[i].set_ylabel(f"{name} (mV)")
axes[1].set_xlabel("Samples")
axes[0].legend(fontsize=8)
fig.suptitle(f"LTDB {ltdb_records[0]} — single Normal beat")
plt.tight_layout()
plt.show()

In [ ]:
# One beat per AAMI class
class_names = {"N": "N — Normal", "S": "S — Supraventricular", 
               "V": "V — Ventricular", "F": "F — Fusion"}

examples = {} # Collect one example per class
for name in ltdb_records:
    # Load signal + annotations
    rec = wfdb.rdrecord(str(LTDB_DIR / name))
    ann = wfdb.rdann(str(LTDB_DIR / name), "atr")
    sig = rec.p_signal
    
    # Find beats and map to AAMI
    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in AAMI_MAP:
            continue
        cls = AAMI_MAP[sym]
        if cls not in examples and r > 100 and r + 150 < len(sig):
            examples[cls] = sig[r-100 : r+150, 0]
    if len(examples) == 4:
        break

n = len(examples)
colors = {"N":"steelblue", "S":"darkorange", "V":"crimson", "F":"green"}

fig, axes = plt.subplots(n, 1, figsize=(11, 2.5*n), sharex=True)
for ax, (cls, beat) in zip(axes, examples.items()):
    ax.plot(beat, lw=0.9, color=colors[cls])
    ax.axvline(100, color="gray", lw=0.6, ls="--")
    ax.set_ylabel(class_names[cls], fontsize=9)

axes[-1].set_xlabel("Samples (128 Hz)")
fig.suptitle("LTDB — one beat per AAMI class (lead MLII)")
plt.tight_layout()
plt.show()

In [ ]:
# PTB-XL metadata
import ast

meta = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)

In [ ]:
print(f"Total records : {len(meta)}")
print(f"Columns       : {list(meta.columns)}\n")

In [ ]:
# PTB-XL superclass distribution
scp_df = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)

def get_superclass(scp_dict):
    out = []
    for code, conf in scp_dict.items():
        if conf >= 50 and code in scp_df.index:
            sc = scp_df.loc[code, "diagnostic_class"]
            if pd.notna(sc):
                out.append(sc)
    return list(set(out))

meta["superclass"] = meta["scp_codes"].apply(get_superclass)

sc_counts = meta["superclass"].explode().value_counts()
print("PTB-XL superclass counts:")
print(sc_counts)

In [ ]:
# One 12-lead record
sample_row = meta.iloc[0]
rec = wfdb.rdrecord(str(PTBXL_DIR / sample_row.filename_lr))
sig = rec.p_signal   # (1000, 12)

lead_names = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
fig, axes = plt.subplots(12, 1, figsize=(13, 14), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(sig[:, i], lw=0.7, color="steelblue")
    ax.set_ylabel(lead_names[i], fontsize=8, rotation=0, labelpad=22)
    ax.axhline(0, color="gray", lw=0.3)
axes[-1].set_xlabel("Samples (100 Hz, 10 s)")
fig.suptitle(f"PTB-XL record {sample_row.name} — {sample_row['superclass']}")
plt.tight_layout(); plt.show()

In [ ]:
# Summary table
print("=" * 45)
print(f"{'':20s}  {'LTDB':>10}  {'PTB-XL':>10}")
print("=" * 45)
print(f"{'Records':20s}  {len(ltdb_records):>10}  {len(meta):>10}")
print(f"{'Leads':20s}  {'2':>10}  {'12':>10}")
print(f"{'Sampling rate':20s}  {'128 Hz':>10}  {'100 Hz':>10}")
print(f"{'Clip length':20s}  {'~20 hrs':>10}  {'10 sec':>10}")
print(f"{'Label type':20s}  {'beat-level':>10}  {'record-level':>10}")
print(f"{'Classes':20s}  {'5 (AAMI)':>10}  {'5 super':>10}")
print("=" * 45)

## Feature Extraction

In [ ]:
from scipy.signal import butter, filtfilt

In [ ]:
# Preprocessing functions
def bandpass_filter(sig, fs, lo=0.5, hi=40.0, order=4):
    """Remove baseline wander (lo) and high-freq noise (hi)."""
    nyq = fs / 2
    b, a = butter(order, [lo/nyq, hi/nyq], btype="band")
    return filtfilt(b, a, sig, axis=0)

def normalize(sig):
    """Z-score per lead so amplitude differences don't dominate."""
    mu  = sig.mean(axis=0)
    std = sig.std(axis=0) + 1e-8
    return (sig - mu) / std

In [ ]:
# Cell 2.3 — visualize filter effect
rec      = wfdb.rdrecord(str(LTDB_DIR / ltdb_records[0]))
sig_raw  = rec.p_signal
sig_filt = bandpass_filter(sig_raw, rec.fs)
t        = 5 * rec.fs   # 5 seconds

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for i, lead in enumerate(rec.sig_name):
    axes[i].plot(sig_raw[:t, i],  lw=0.7, color="gray",      label="raw")
    axes[i].plot(sig_filt[:t, i], lw=0.8, color="steelblue", label="filtered")
    axes[i].set_ylabel(f"{lead} (mV)")
    axes[i].legend(fontsize=8)
axes[1].set_xlabel("Samples")
fig.suptitle("Bandpass filter effect (0.5–40 Hz)")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 2.4 — LTDB beat extraction
AAMI_MAP   = {
    "N":"N","L":"N","R":"N","e":"N","j":"N",
    "A":"S","a":"S","J":"S","S":"S",
    "V":"V","E":"V",
    "F":"F",
}
AAMI_LABEL = {"N":0, "S":1, "V":2, "F":3}
WIN_PRE, WIN_POST = 100, 150   # 250 samples total around R-peak

# Cell 2.4 — fixed extract_beats
def extract_beats(record_name):
    rec = wfdb.rdrecord(str(LTDB_DIR / record_name))
    ann = wfdb.rdann(str(LTDB_DIR / record_name), "atr")
    sig = rec.p_signal.astype(np.float32)
    sig = sig[:, :2]                        # ← keep only first 2 leads
    sig = bandpass_filter(sig, rec.fs)
    sig = normalize(sig)

    beats, labels = [], []
    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in AAMI_MAP: continue
        if r < WIN_PRE or r + WIN_POST > len(sig): continue
        beats.append(sig[r-WIN_PRE : r+WIN_POST].T)  # (2, 250)
        labels.append(AAMI_LABEL[AAMI_MAP[sym]])
    return np.stack(beats), np.array(labels)

# re-run extraction
ltdb_X, ltdb_y = [], []
for name in ltdb_records:
    X, y = extract_beats(name)
    ltdb_X.append(X); ltdb_y.append(y)
    print(f"  {name}: {X.shape} — {Counter(y)}")

ltdb_X = np.concatenate(ltdb_X)
ltdb_y = np.concatenate(ltdb_y)
print(f"\nLTDB total: {ltdb_X.shape} | labels: {Counter(ltdb_y)}")

In [ ]:
# Cell 2.5 — PTB-XL extraction
meta = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)
scp_df = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)

def get_superclass(scp_dict):
    out = []
    for code, conf in scp_dict.items():
        if conf >= 50 and code in scp_df.index:
            sc = scp_df.loc[code, "diagnostic_class"]
            if pd.notna(sc): out.append(sc)
    return list(set(out))

meta["superclass"] = meta["scp_codes"].apply(get_superclass)
SC_MAP = {"NORM":0, "MI":1, "STTC":2, "CD":3, "HYP":4}

def load_ptbxl(row):
    rec = wfdb.rdrecord(str(PTBXL_DIR / row.filename_lr))
    sig = rec.p_signal.astype(np.float32)   # (1000, 12)
    sig = bandpass_filter(sig, 100)
    sig = normalize(sig)
    return sig.T                             # (12, 1000)

ptbxl_X, ptbxl_y = [], []
for ecg_id, row in meta.iterrows():
    scs = row["superclass"]
    if len(scs) != 1: continue
    label = SC_MAP.get(scs[0], -1)
    if label < 0: continue
    ptbxl_X.append(load_ptbxl(row))
    ptbxl_y.append(label)

ptbxl_X = np.stack(ptbxl_X)
ptbxl_y = np.array(ptbxl_y)
print(f"PTB-XL total: {ptbxl_X.shape} | labels: {Counter(ptbxl_y)}")

In [ ]:
# Cell 2.6 — visualize preprocessed beats per class
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
class_names = {0:"N — Normal", 1:"S — Supraventricular",
               2:"V — Ventricular", 3:"F — Fusion"}
colors = {0:"steelblue", 1:"darkorange", 2:"crimson", 3:"green"}

for cls in range(4):
    idx = np.where(ltdb_y == cls)[0][0]
    axes[cls].plot(ltdb_X[idx, 0], lw=0.9, color=colors[cls])
    axes[cls].axvline(100, color="gray", lw=0.6, ls="--", label="R-peak")
    axes[cls].set_ylabel(class_names[cls], fontsize=9)
    axes[cls].legend(fontsize=7)

axes[-1].set_xlabel("Samples (128 Hz, normalized)")
fig.suptitle("LTDB — preprocessed beats per class")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 2.7 — save arrays
import os
os.makedirs("../processed_data", exist_ok=True)

np.save("../processed_data/ltdb_X.npy", ltdb_X)
np.save("../processed_data/ltdb_y.npy", ltdb_y)
np.save("../processed_data/ptbxl_X.npy", ptbxl_X)
np.save("../processed_data/ptbxl_y.npy", ptbxl_y)
print("Saved. Shapes:")
print(f"  ltdb_X : {ltdb_X.shape}")
print(f"  ptbxl_X: {ptbxl_X.shape}")

## Model definition

In [ ]:
# Cell 3.1 — model
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.conv(x) + self.skip(x))


class ECGNet(nn.Module):
    """
    CNN extracts local beat morphology.
    Transformer captures temporal context across the signal.
    Two separate heads for LTDB and PTB-XL.
    """
    def __init__(self, in_ch=12, d_model=128, n_heads=4,
                 n_layers=4, n_cls_a=4, n_cls_b=5):
        super().__init__()

        # CNN encoder
        self.cnn = nn.Sequential(
            ConvBlock(in_ch, 32,      kernel=7),
            nn.MaxPool1d(2),
            ConvBlock(32,   64,      kernel=5),
            nn.MaxPool1d(2),
            ConvBlock(64,   d_model, kernel=3),
        )

        # Positional encoding
        self.pos_drop = nn.Dropout(0.1)
        self._d_model = d_model

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Transformer
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=256,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers,
                                                  norm=nn.LayerNorm(d_model))
        # Heads
        self.head_ltdb  = nn.Linear(d_model, n_cls_a)
        self.head_ptbxl = nn.Linear(d_model, n_cls_b)

    def _pos_enc(self, T):
        pos = torch.arange(T).unsqueeze(1)
        div = torch.exp(torch.arange(0, self._d_model, 2) *
                        (-math.log(10000.0) / self._d_model))
        pe  = torch.zeros(T, self._d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)   # (1, T, d_model)

    def encode(self, x):
        z   = self.cnn(x)                              # (B, d_model, T')
        z   = z.permute(0, 2, 1)                       # (B, T', d_model)
        pe  = self._pos_enc(z.size(1)).to(x.device)
        z   = self.pos_drop(z + pe)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        z   = torch.cat([cls, z], dim=1)               # prepend CLS
        z   = self.transformer(z)
        return z[:, 0]                                 # CLS embedding

    def forward(self, x, dataset="ltdb"):
        h = self.encode(x)
        if dataset == "ltdb":
            return self.head_ltdb(h)
        else:
            return self.head_ptbxl(h)

In [ ]:
# Cell 3.2 — quick shape test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# test LTDB shape: (batch, 2 leads, 250 samples)
model = ECGNet(in_ch=2, n_cls_a=4, n_cls_b=5).to(device)
x_test = torch.randn(8, 2, 250).to(device)
out    = model(x_test, dataset="ltdb")
print("LTDB output shape:", out.shape)   # (8, 4)

# test PTB-XL shape: (batch, 12 leads, 1000 samples)
model2 = ECGNet(in_ch=12, n_cls_a=4, n_cls_b=5).to(device)
x_test2 = torch.randn(8, 12, 1000).to(device)
out2    = model2(x_test2, dataset="ptbxl")
print("PTB-XL output shape:", out2.shape)  # (8, 5)

## Training

In [ ]:
# Cell 4.1 — data loaders
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

class ECGDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.augment:
            x += torch.randn_like(x) * 0.05          # small gaussian noise
            x *= np.random.uniform(0.9, 1.1)          # amplitude jitter
        return x, self.y[i]


def make_loaders(X, y, batch=64):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15,
                                           stratify=y, random_state=42)
    Xtr, Xval, ytr, yval = train_test_split(Xtr, ytr, test_size=0.15,
                                             stratify=ytr, random_state=42)
    # weighted sampler — fixes class imbalance
    counts  = np.bincount(ytr)
    weights = 1.0 / counts[ytr]
    sampler = WeightedRandomSampler(weights, len(weights))

    tr  = DataLoader(ECGDataset(Xtr,  ytr,  augment=True), batch_size=batch, sampler=sampler)
    val = DataLoader(ECGDataset(Xval, yval), batch_size=batch, shuffle=False)
    te  = DataLoader(ECGDataset(Xte,  yte),  batch_size=batch, shuffle=False)
    return tr, val, te, (Xte, yte)

tr_ltdb,  val_ltdb,  te_ltdb,  (Xte_l, yte_l) = make_loaders(ltdb_X,  ltdb_y)
tr_ptbxl, val_ptbxl, te_ptbxl, (Xte_p, yte_p) = make_loaders(ptbxl_X, ptbxl_y)
print("LTDB  — train/val/test splits done")
print("PTB-XL — train/val/test splits done")

In [ ]:
# Cell 4.2 — training function
from sklearn.metrics import f1_score, classification_report

def train_model(model, tr_loader, val_loader, dataset="ltdb",
                epochs=40, lr=3e-4):
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_f1, best_state = 0, None
    history = {"train_loss": [], "val_f1": []}

    for ep in range(epochs):
        # train
        model.train()
        total_loss = 0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x, dataset), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
            total_loss += loss.item()

        # validate
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for x, y in val_loader:
                p = model(x.to(device), dataset).argmax(1).cpu()
                preds.append(p); trues.append(y)
        val_f1 = f1_score(torch.cat(trues), torch.cat(preds), average="macro")

        history["train_loss"].append(total_loss / len(tr_loader))
        history["val_f1"].append(val_f1)
        sched.step()

        if val_f1 > best_f1:
            best_f1  = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (ep+1) % 5 == 0:
            print(f"  Ep {ep+1:02d} | loss={history['train_loss'][-1]:.4f} | val_F1={val_f1:.4f}")

    model.load_state_dict(best_state)
    print(f"\nBest val F1: {best_f1:.4f}")
    return history

In [ ]:
# Cell 4.3 — train LTDB model
print("=== Training LTDB model ===")
model_ltdb = ECGNet(in_ch=2, n_cls_a=4, n_cls_b=5).to(device)
hist_ltdb  = train_model(model_ltdb, tr_ltdb, val_ltdb, dataset="ltdb", epochs=40)

In [ ]:
# Cell 4.4 — train PTB-XL model
print("=== Training PTB-XL model ===")
model_ptbxl = ECGNet(in_ch=12, n_cls_a=4, n_cls_b=5).to(device)
hist_ptbxl  = train_model(model_ptbxl, tr_ptbxl, val_ptbxl, dataset="ptbxl", epochs=40)

In [ ]:
# Cell 4.5 — plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, title in zip(axes,
                            [hist_ltdb, hist_ptbxl],
                            ["LTDB", "PTB-XL"]):
    ax2 = ax.twinx()
    ax.plot(hist["train_loss"], color="steelblue", label="train loss")
    ax2.plot(hist["val_f1"],    color="crimson",   label="val F1")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss", color="steelblue")
    ax2.set_ylabel("Val F1", color="crimson")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.6 — test evaluation
def evaluate(model, loader, dataset, class_names):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            p = model(x.to(device), dataset).argmax(1).cpu()
            preds.append(p); trues.append(y)
    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(trues).numpy()
    print(classification_report(y_true, y_pred, target_names=class_names))
    return y_pred, y_true

print("=== LTDB Test Results ===")
pred_l, true_l = evaluate(model_ltdb, te_ltdb, "ltdb",
                           ["N","S","V","F"])
print("\n=== PTB-XL Test Results ===")
pred_p, true_p = evaluate(model_ptbxl, te_ptbxl, "ptbxl",
                           ["NORM","MI","STTC","CD","HYP"])

In [ ]:
# Cell 4.7 — confusion matrices
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_true, y_pred, names, title in zip(
        axes,
        [true_l, true_p],
        [pred_l, pred_p],
        [["N","S","V","F"], ["NORM","MI","STTC","CD","HYP"]],
        ["LTDB", "PTB-XL"]):
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
plt.tight_layout(); plt.show()

## Interpretability

In [ ]:
# Cell 5.1 — attention rollout (what time steps does model focus on?)
def get_attention_rollout(model, x_input, dataset="ltdb"):
    model.eval()
    x = torch.tensor(x_input[np.newaxis], dtype=torch.float32).to(device)

    # get CNN output length
    with torch.no_grad():
        cnn_out = model.cnn(x)
    T = cnn_out.shape[2]

    # hook attention weights from each layer
    attn_maps = []
    hooks = []
    def make_hook(attn_list):
        def hook(module, inp, out):
            # out is (attn_output, attn_weights) when need_weights=True
            pass
        return hook

    # simpler: rerun with manual extraction
    z = cnn_out.permute(0, 2, 1)
    pe = model._pos_enc(T).to(device)
    z  = model.pos_drop(z + pe)
    cls = model.cls_token.expand(1, -1, -1)
    z   = torch.cat([cls, z], dim=1)

    rollout = torch.eye(T + 1).to(device)
    with torch.no_grad():
        for layer in model.transformer.layers:
            attn_out, attn_w = layer.self_attn(z, z, z,
                                                need_weights=True,
                                                average_attn_weights=True)
            A = attn_w[0] + torch.eye(T+1).to(device)
            A = A / A.sum(dim=-1, keepdim=True)
            rollout = A @ rollout
            z = layer(z)

    cls_attn = rollout[0, 1:].cpu().numpy()   # (T,) attention to each token
    return cls_attn

# plot attention on a beat
sample = ltdb_X[np.where(ltdb_y == 2)[0][0]]   # pick a V beat
attn   = get_attention_rollout(model_ltdb, sample, dataset="ltdb")

# upsample attention to signal length
from scipy.ndimage import zoom
attn_full = zoom(attn, sample.shape[1] / len(attn))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(sample[0], lw=0.9, color="crimson", label="Lead MLII (V beat)")
axes[0].axvline(100, color="gray", lw=0.6, ls="--", label="R-peak")
axes[0].set_ylabel("Amplitude"); axes[0].legend(fontsize=8)
axes[1].fill_between(np.arange(len(attn_full)), attn_full,
                     color="darkorange", alpha=0.7)
axes[1].set_ylabel("Attention weight")
axes[1].set_xlabel("Samples")
fig.suptitle("Transformer attention rollout — Ventricular beat")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 5.2 — Grad-CAM (which part of signal drives the prediction?)
def grad_cam(model, x_input, target_class, dataset="ltdb"):
    model.eval()
    x = torch.tensor(x_input[np.newaxis], dtype=torch.float32).to(device)
    x.requires_grad_(True)

    logits = model(x, dataset)
    model.zero_grad()
    logits[0, target_class].backward()

    grad = x.grad.detach().cpu().numpy()[0]   # (leads, T)
    cam  = np.abs(grad).mean(axis=0)          # average over leads
    cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

# one sample per class
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
class_names = ["N","S","V","F"]
colors = ["steelblue","darkorange","crimson","green"]

for cls in range(4):
    idx    = np.where(ltdb_y == cls)[0][0]
    sample = ltdb_X[idx]
    cam    = grad_cam(model_ltdb, sample, cls)

    ax = axes[cls]
    ax.plot(sample[0], lw=0.8, color=colors[cls], label=class_names[cls])
    ax.fill_between(np.arange(len(cam)), 
                    sample[0].min() * cam,
                    sample[0].max() * cam,
                    alpha=0.35, color="gold", label="Grad-CAM")
    ax.axvline(100, color="gray", lw=0.5, ls="--")
    ax.set_ylabel(class_names[cls], fontsize=9)
    ax.legend(fontsize=7, loc="upper right")

axes[-1].set_xlabel("Samples")
fig.suptitle("Grad-CAM — model focus per class (LTDB)")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 5.3 — SHAP (feature importance over time)
import shap

model_ltdb.eval()

# background = 100 random N beats
bg_idx = np.random.choice(np.where(ltdb_y == 0)[0], 100, replace=False)
bg     = torch.tensor(ltdb_X[bg_idx], dtype=torch.float32).to(device)

# test = 20 beats (5 per class)
test_idx = []
for cls in range(4):
    idx = np.where(ltdb_y == cls)[0][:5]
    test_idx.extend(idx)
test_x = torch.tensor(ltdb_X[test_idx], dtype=torch.float32).to(device)

# wrap model for shap
def model_fn(x):
    with torch.no_grad():
        out = model_ltdb(torch.tensor(x, dtype=torch.float32).to(device), "ltdb")
    return F.softmax(out, dim=-1).cpu().numpy()

explainer   = shap.GradientExplainer(model_ltdb, bg)
shap_values = explainer.shap_values(test_x)
# shap_values: list of 4 arrays, each (20, 2, 250)

# plot mean absolute SHAP over time for each class
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for cls in range(4):
    sv  = np.abs(shap_values[cls]).mean(axis=(0, 1))  # avg over samples & leads → (250,)
    axes[cls].fill_between(np.arange(250), sv, alpha=0.7, color=colors[cls])
    axes[cls].set_ylabel(f"Class {class_names[cls]}", fontsize=9)
    axes[cls].axvline(100, color="gray", lw=0.5, ls="--")

axes[-1].set_xlabel("Samples (R-peak at 100)")
fig.suptitle("SHAP — mean absolute feature importance over time (LTDB)")
plt.tight_layout(); plt.show()

## Generalibility

# Cell 6.1 — noise robustness test

In [ ]:
def eval_f1(model, X, y, dataset, noise_std=0.0):
    model.eval()
    loader = DataLoader(ECGDataset(X + np.random.randn(*X.shape).astype(np.float32) * noise_std, y),
                        batch_size=64, shuffle=False)
    preds, trues = [], []
    with torch.no_grad():
        for x, yb in loader:
            p = model(x.to(device), dataset).argmax(1).cpu()
            preds.append(p); trues.append(yb)
    return f1_score(torch.cat(trues), torch.cat(preds), average="macro")

noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]

f1_ltdb  = [eval_f1(model_ltdb,  Xte_l, yte_l, "ltdb",  s) for s in noise_levels]
f1_ptbxl = [eval_f1(model_ptbxl, Xte_p, yte_p, "ptbxl", s) for s in noise_levels]

plt.figure(figsize=(8, 4))
plt.plot(noise_levels, f1_ltdb,  "o-", color="steelblue", label="LTDB")
plt.plot(noise_levels, f1_ptbxl, "s-", color="crimson",   label="PTB-XL")
plt.xlabel("Gaussian noise σ"); plt.ylabel("Macro F1")
plt.title("Model robustness to additive noise")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nLTDB  noise robustness:")
for s, f in zip(noise_levels, f1_ltdb):  print(f"  σ={s:.2f} → F1={f:.4f}")
print("\nPTB-XL noise robustness:")
for s, f in zip(noise_levels, f1_ptbxl): print(f"  σ={s:.2f} → F1={f:.4f}")

In [ ]:
# Cell 6.2 — OOD test: MIT-BIH Arrhythmia DB (completely unseen)
# Same AAMI label scheme as LTDB but different patients + equipment
MITDB_DIR = Path("../raw_data/mit-bih-arrhythmia-database-1.0.0")

# download if not present
if not MITDB_DIR.exists():
    MITDB_DIR.mkdir(parents=True)
    wfdb.dl_database("mitdb", str(MITDB_DIR))

mitdb_records = [p.stem for p in MITDB_DIR.glob("*.hea")
                 if not p.name.endswith(".hea-")]
print(f"MIT-BIH records: {len(mitdb_records)}")

In [ ]:
# Cell 6.3 — extract MIT-BIH beats (same pipeline as LTDB)
def extract_beats_mitdb(record_name):
    rec = wfdb.rdrecord(str(MITDB_DIR / record_name))
    ann = wfdb.rdann(str(MITDB_DIR / record_name), "atr")
    sig = rec.p_signal.astype(np.float32)

    # MIT-BIH is 360 Hz — resample to 128 Hz
    from scipy.signal import resample
    target_len = int(sig.shape[0] * 128 / rec.fs)
    sig = resample(sig, target_len, axis=0)
    scale = 128 / rec.fs
    r_peaks = (ann.sample * scale).astype(int)

    sig = bandpass_filter(sig, 128)
    sig = normalize(sig)

    beats, labels = [], []
    for r, sym in zip(r_peaks, ann.symbol):
        if sym not in AAMI_MAP: continue
        if r < WIN_PRE or r + WIN_POST > len(sig): continue
        beats.append(sig[r-WIN_PRE : r+WIN_POST].T)
        labels.append(AAMI_LABEL[AAMI_MAP[sym]])
    return np.stack(beats), np.array(labels)

ood_X, ood_y = [], []
for name in mitdb_records:
    try:
        X, y = extract_beats_mitdb(name)
        ood_X.append(X); ood_y.append(y)
    except Exception as e:
        print(f"  skip {name}: {e}")

ood_X = np.concatenate(ood_X)
ood_y = np.concatenate(ood_y)
print(f"OOD dataset: {ood_X.shape} | {Counter(ood_y)}")

In [ ]:
# Cell 6.4 — evaluate on OOD data
ood_loader = DataLoader(ECGDataset(ood_X, ood_y), batch_size=64, shuffle=False)

print("=== OOD (MIT-BIH) — model trained only on LTDB ===")
pred_ood, true_ood = evaluate(model_ltdb, ood_loader, "ltdb", ["N","S","V","F"])

ood_f1 = f1_score(true_ood, pred_ood, average="macro")
in_f1  = f1_score(true_l,   pred_l,   average="macro")
print(f"\nIn-distribution F1 (LTDB test) : {in_f1:.4f}")
print(f"Out-of-distribution F1 (MIT-BIH): {ood_f1:.4f}")
print(f"F1 drop                          : {in_f1 - ood_f1:.4f}")